In [7]:
import os
import requests
from dotenv import load_dotenv
import pandas as pd

In [3]:
load_dotenv()
HEADERS = {"User-Agent": os.getenv("SEC_USER_AGENT")} 

cik = "66740"
url = f"https://data.sec.gov/api/xbrl/companyfacts/CIK{int(cik):010d}.json"

responde =requests.get(url,headers=HEADERS,timeout=30)

print(responde.status_code)   # 200 = tutto ok

data = responde.json()
print(data["entityName"])
print(len(data["facts"]["us-gaap"]), "concetti us-gaap")

200
3M COMPANY
706 concetti us-gaap


In [5]:
print(len(data["facts"]["us-gaap"]["PaymentsToAcquirePropertyPlantAndEquipment"]["units"]["USD"]))

181


In [8]:
df=pd.DataFrame(data["facts"]["us-gaap"]["PaymentsToAcquirePropertyPlantAndEquipment"]["units"]["USD"])
df.head()

,start,end,val,accn,fy,fp,form,filed,frame
0,2007-01-01,2007-12-31,1422000000,0001104659-10-007295,2009.0,FY,10-K,2010-02-16,NaN
1,2007-01-01,2007-12-31,1422000000,0001104659-10-029054,2009.0,FY,8-K,2010-05-17,CY2007
2,2008-01-01,2008-06-30,632000000,0001104659-09-046329,2009.0,Q2,10-Q,2009-07-31,NaN
3,2008-01-01,2008-09-30,1008000000,0001104659-09-061503,2009.0,Q3,10-Q,2009-10-30,NaN
4,2008-01-01,2008-12-31,1471000000,0001104659-10-007295,2009.0,FY,10-K,2010-02-16,NaN


In [9]:
df[df["end"]=="2018-12-31"]

,start,end,val,accn,fy,fp,form,filed,frame
112,2018-01-01,2018-12-31,1577000000,0001558370-19-000470,2018.0,FY,10-K,2019-02-07,NaN
113,2018-01-01,2018-12-31,1577000000,0001558370-20-000581,2019.0,FY,10-K,2020-02-06,NaN
114,2018-01-01,2018-12-31,1577000000,0001558370-21-000737,2020.0,FY,10-K,2021-02-04,CY2018


In [13]:
rows = []

for concept, info in data["facts"]["us-gaap"].items():
    for unit,observations  in info["units"].items():
        for obs in observations:
            rows.append({"concept": concept, "unit": unit, **obs})

In [14]:
df_3m = pd.DataFrame(rows)
print(df_3m.shape)

(35633, 11)


In [ ]:
df_3m[
    (df_3m["concept"] == "PaymentsToAcquirePropertyPlantAndEquipment")
    & (df_3m["end"] == "2018-12-31")
]

,concept,unit,end,val,accn,fy,fp,form,filed,frame,start
27040,PaymentsToAcquirePropertyPlantAndEquipment,USD,2018-12-31,1.577000e+09,0001558370-19-000470,2018.0,FY,10-K,2019-02-07,NaN,2018-01-01
27041,PaymentsToAcquirePropertyPlantAndEquipment,USD,2018-12-31,1.577000e+09,0001558370-20-000581,2019.0,FY,10-K,2020-02-06,NaN,2018-01-01
27042,PaymentsToAcquirePropertyPlantAndEquipment,USD,2018-12-31,1.577000e+09,0001558370-21-000737,2020.0,FY,10-K,2021-02-04,CY2018,2018-01-01


In [1]:
import json
import os
import time
from pathlib import Path

import requests
from dotenv import load_dotenv

from finrag.config import get_companies

mapping = json.loads(Path("../data/processed/company_cik.json").read_text(encoding="utf-8"))

# 1. I codici coincidono con quelli dell'indice?
index_companies = set(get_companies())
print("Nell'indice ma non nella tabella:", sorted(index_companies - set(mapping)))
print("Nella tabella ma non nell'indice:", sorted(set(mapping) - index_companies))
print()


c:\Users\domen\Desktop\Agenti\Progetto\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Nell'indice ma non nella tabella: []
Nella tabella ma non nell'indice: []



### File preparation (use this code for creating the xbrl file)

In [2]:
from finrag.xbrl import flatten_comapnyfacts, RAW_DIR,PROCESSED_DIR, clean_annual_facts
import pandas as pd
import json


frames=[]
for path in RAW_DIR.glob("*.json"):
    data = json.loads(path.read_text(encoding="utf-8"))
    frames.append(flatten_comapnyfacts(data=data, company=path.stem))

df_facts = pd.concat(frames, ignore_index=True)
out = PROCESSED_DIR / "xbrl_facts.parquet"

In [3]:
print(df_facts.shape)
print(df_facts["company"].nunique())
print(df_facts.isna().sum())

(1125858, 12)
41
company         0
concept         0
unit            0
end             0
val             0
accn            0
fy          11084
fp          11084
form            0
filed           0
frame      672963
start      424010
dtype: int64


In [ ]:
df_annual = clean_annual_facts(df_facts)
print(df_annual.shape)

print(df_annual[
    (df_annual["company"] == "3M")
    & (df_annual["concept"] == "PaymentsToAcquirePropertyPlantAndEquipment")
    & (df_annual["year"] == 2018)
])

(183817, 13)
     company                                     concept  ...      start  year
4028      3M  PaymentsToAcquirePropertyPlantAndEquipment  ... 2018-01-01  2018

[1 rows x 13 columns]


In [7]:
df_annual.to_parquet(PROCESSED_DIR / "xbrl_facts_annual.parquet", index=False)

In [1]:
from finrag.xbrl import flatten_comapnyfacts, RAW_DIR,PROCESSED_DIR, get_metric
import pandas as pd
import json
print(get_metric("3M", "capex", 2018))                     # 1577000000.0, PaymentsToAcquirePropertyPlantAndEquipment
print(get_metric("ADOBE", "current_liabilities", 2015))    # 2213556000.0
print(get_metric("CORNING", "capex", 2018))                # None? Era nelle lacune
get_metric("3M", "fatturato", 2018)                        # deve dare ValueError

{'company': '3M', 'metric': 'capex', 'year': 2018, 'value': 1577000000.0, 'unit': 'USD', 'concept': 'PaymentsToAcquirePropertyPlantAndEquipment', 'period_end': '2018-12-31', 'filed': '2019-02-07', 'form': '10-K'}
{'company': 'ADOBE', 'metric': 'current_liabilities', 'year': 2015, 'value': 2213556000.0, 'unit': 'USD', 'concept': 'LiabilitiesCurrent', 'period_end': '2015-11-27', 'filed': '2016-01-19', 'form': '10-K'}
None


ValueError: Unknown metric 'fatturato'. Allowed: ['accounts_payable', 'accounts_receivable', 'capex', 'cogs', 'current_assets', 'current_liabilities', 'depreciation_amortization', 'dividends_paid', 'inventory', 'net_income', 'net_income_incl_nci', 'operating_cash_flow', 'operating_income', 'ppe_net', 'revenue', 'total_assets']

In [1]:
from pathlib import Path
import pandas as pd

from finrag.xbrl import get_metric, get_metric_vocabulary
from finrag.ingestion import parse_doc_name   # <- adatta al modulo dove hai parse_doc_name

PDF_DIR = Path("../data/raw/pdfs")

# Le coppie azienda-anno dei documenti annuali del corpus
pairs = set()
for pdf in PDF_DIR.glob("*.pdf"):
    parsed = parse_doc_name(pdf.stem)
    if parsed and parsed["doc_type"] in {"10K", "ANNUALREPORT"}:
        pairs.add((parsed["company"], parsed["fiscal_year"]))
pairs = sorted(pairs)
print(f"Coppie azienda-anno annuali nel corpus: {len(pairs)}\n")

rows = []
for metric in get_metric_vocabulary():
    for company, year in pairs:
        rows.append({
            "metric": metric,
            "company": company,
            "year": year,
            "found": get_metric(company, metric, year) is not None,
        })
coverage = pd.DataFrame(rows)

# Copertura per metrica
summary = coverage.groupby("metric")["found"].agg(["mean", "sum", "count"])
summary.columns = ["copertura", "trovati", "totale"]
print(summary.sort_values("copertura").round(3), "\n")

# Le lacune, raggruppate per metrica e azienda
missing = coverage[~coverage["found"]]
print("Lacune (metrica → aziende e anni):")
for metric, group in missing.groupby("metric"):
    by_company = group.groupby("company")["year"].apply(lambda ys: sorted(ys))
    print(f"\n{metric}:")
    for company, years in by_company.items():
        print(f"  {company:22s} {years}")

c:\Users\domen\Desktop\Agenti\Progetto\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Coppie azienda-anno annuali nel corpus: 270

                           copertura  trovati  totale
metric                                               
dividends_paid                 0.733      198     270
depreciation_amortization      0.789      213     270
cogs                           0.800      216     270
inventory                      0.804      217     270
operating_income               0.867      234     270
accounts_receivable            0.922      249     270
capex                          0.944      255     270
revenue                        0.963      260     270
accounts_payable               0.981      265     270
current_liabilities            0.985      266     270
current_assets                 0.985      266     270
ppe_net                        0.996      269     270
operating_cash_flow            1.000      270     270
net_income_incl_nci            1.000      270     270
net_income                     1.000      270     270
total_assets                   1.000 

In [1]:
import re
import pandas as pd
from finrag.xbrl import get_metric

df = pd.read_json("../data/processed/financebench_p.json", lines=True)


class Missing(Exception):
    """Un dato richiesto dalla formula non c'è in XBRL."""

def v(company, metric, year):
    result = get_metric(company, metric, year)
    if result is None:
        raise Missing(f"{metric} {company} {year}")
    return result["value"]

def avg(*values):
    return sum(values) / len(values)

def first_number(text):
    """Il primo numero di una stringa, ignorando $, virgole e %."""
    match = re.search(r"-?\d[\d,]*\.?\d*", str(text))
    return float(match.group().replace(",", "")) if match else None


# financebench_id -> formula (in USD milioni per gli importi, numero puro per i rapporti)
CASES = {
    # 3M: capex FY2018, USD milioni
    "financebench_id_03029": lambda: v("3M", "capex", 2018) / 1e6,
    # Adobe: operating cash flow ratio FY2015
    "financebench_id_04735": lambda: v("ADOBE", "operating_cash_flow", 2015) / v("ADOBE", "current_liabilities", 2015),
    # Lockheed Martin: asset turnover FY2020 = ricavi / media del totale attivo 2019-2020
    "financebench_id_04412": lambda: v("LOCKHEEDMARTIN", "revenue", 2020)
        / avg(v("LOCKHEEDMARTIN", "total_assets", 2019), v("LOCKHEEDMARTIN", "total_assets", 2020)),
    # AES: rotazione del magazzino FY2022 = costo del venduto / media del magazzino 2021-2022
    "financebench_id_00540": lambda: v("AES", "cogs", 2022)
        / avg(v("AES", "inventory", 2021), v("AES", "inventory", 2022)),
    # Activision Blizzard: capex medio FY2017-FY2019, USD milioni
    "financebench_id_07966": lambda: avg(*(v("ACTIVISIONBLIZZARD", "capex", y) for y in (2017, 2018, 2019))) / 1e6,
}

results = []
for fb_id, formula in CASES.items():
    row = df[df["financebench_id"] == fb_id].iloc[0]
    reference = first_number(row["answer"])
    try:
        value = formula()
        ok = reference is not None and abs(value - reference) <= 0.01 * abs(reference)
        outcome = "corretto" if ok else "SBAGLIATO"
    except Missing as e:
        value, outcome = None, f"dato mancante: {e}"
    results.append({
        "id": fb_id,
        "domanda": row["question"][:70],
        "xbrl": None if value is None else round(value, 2),
        "riferimento": reference,
        "esito": outcome,
    })

pd.set_option("display.max_colwidth", 80)
print(pd.DataFrame(results))

c:\Users\domen\Desktop\Agenti\Progetto\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


                      id  \
0  financebench_id_03029   
1  financebench_id_04735   
2  financebench_id_04412   
3  financebench_id_00540   
4  financebench_id_07966   

                                                                  domanda  \
0  What is the FY2018 capital expenditure amount (in USD millions) for 3M   
1  You are an investment banker and your only resource(s) to answer the f   
2  We need to calculate a reasonable approximation (or exact number if po   
3  Roughly how many times has AES Corporation sold its inventory in FY202   
4  What is the FY2017 - FY2019 3 year average of capex as a % of revenue    

      xbrl  riferimento      esito  
0  1577.00      1577.00   corretto  
1     0.66         0.66   corretto  
2     1.33         1.33   corretto  
3    12.14         9.50  SBAGLIATO  
4   134.00         1.90  SBAGLIATO  
